# MARSNet — EKF Baseline

## What this notebook implements
A 6-state linear Kalman Filter as the classical dead-reckoning baseline.
No learned components. GPS-aided during normal flight, pure IMU propagation during outages.

## State vector
```
x = [vx, vy, vz, bx, by, bz]   (velocity NED + accelerometer bias body-frame)
```

## Process model (dt = 0.1s)
```
v[t+1] = v[t] + (a_measured[t] - b[t]) * dt
b[t+1] = b[t]   (bias = random walk)
```
Linear process model → standard KF (not EKF) suffices.

## Measurement model (GPS-aided windows only)
```
z = v_GPS    H = [I_3 | 0_3]    R ≈ diag(0.0001)  (RTK: ~0.01 m/s per axis)
```

## Tuning protocol
1. Run with sensor-spec defaults (untuned) — report these numbers.
2. Tune Q_v, Q_bias, R on val set via Nelder-Mead (same protocol as LAM_CVPRIOR search).
3. Report both. Do NOT tune on test set.

## Expected per-group pattern (itself a paper finding)
| Group | Expected | Why |
|-------|----------|-----|
| S0–S34 straight | Low drift | Near-constant v — ideal for KF with tuned Q |
| S35–S40 straight-med | Low drift | Same |
| S41–S46 turns | Degraded | No coordinated-turn kinematic term in process model |
| S47–S52 false-alarm | Low drift | GPS actually constant — KF propagates correctly |
| S53–S58 long outage | Degrades with length | Unbounded P growth without GPS updates |

This is the 3-way narrative: naive fails on turns, EKF fails on turns,
MARSNet succeeds on turns but fails on S47–S52 where classical methods succeed trivially.

## R20 reference numbers
```
MARSNet R20: mean=6.56m  median=0.48m  90th=32.52m  79.7%<5m
Naive:       mean=5.96m  median=0.38m
S41-46: MARSNet=2.97m  Naive=33.78m
S47-52: MARSNet=32.56m Naive=0.66m
```

In [ ]:
import os, math, time, warnings
import numpy as np
import pandas as pd
from scipy.optimize import minimize
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
print('Imports OK')

In [ ]:
# ── Configuration (portable — no Colab / Google Drive required) ───────────────
import os
DATA_PATH = os.environ.get("GATEIO_DATA", "../data/processed/MARS_Master_Dataset.npz")
PLOT_DIR  = os.environ.get("GATEIO_OUT",  "./results")
os.makedirs(PLOT_DIR, exist_ok=True)
print(f"Data:  {DATA_PATH}")
print(f"Plots: {PLOT_DIR}")


In [ ]:
# ── Constants — must match MARSNet exactly ─────────────────────────────────────
SEQ_LEN = 300
WIN_LEN = 200
DT      = 0.1     # seconds per window

# Outage window position — identical to MARSNet simulation
OS_FRAC = 1 / 3   # outage starts at SEQ_LEN // 3 = window 100
OE_LEN  = 100     # outage duration: 100 windows = 10 seconds

# Sequence groups — identical to MARSNet evaluation
GROUP_MAP = {**{i: 'straight-short' for i in range(0, 35)},
             **{i: 'straight-med'   for i in range(35, 41)},
             **{i: 'TURN'           for i in range(41, 47)},
             **{i: 'FALSE-ALARM'    for i in range(47, 53)},
             **{i: 'long-outage'    for i in range(53, 59)}}
GROUP_ORDER = ['straight-short', 'straight-med', 'TURN', 'FALSE-ALARM', 'long-outage']

# ── Initial (untuned) KF noise parameters ─────────────────────────────────────
# R: RTK GPS velocity accuracy ~0.01 m/s → R = 0.01^2 = 1e-4 per axis
R_INIT    = 1e-4
# Q_v: accelerometer integration noise
#   DJI M300 IMU accel noise density ≈ 150 μg/√Hz = 1.47e-3 m/s²/√Hz
#   Discrete: σ_a²*dt² = (1.47e-3)^2 * 0.1^2 ≈ 2.16e-8 → round up for robustness
Q_V_INIT  = 1e-4
# Q_bias: accelerometer bias random walk
#   Typical in-run bias instability: ~0.1 mg/s → small
Q_B_INIT  = 1e-6

# Search bounds for Nelder-Mead (in log-space)
# log(1e-8) to log(1e-1) covers 7 orders of magnitude
LOG_BOUNDS = [(-8., -1.),   # log Q_v
              (-12., -3.),  # log Q_bias
              (-8., -1.)]   # log R

print('Config OK')
print(f'  DT={DT}s  SEQ_LEN={SEQ_LEN}  outage=[{int(SEQ_LEN*OS_FRAC)}, {int(SEQ_LEN*OS_FRAC)+OE_LEN})')
print(f'  Untuned: Q_v={Q_V_INIT:.0e}  Q_bias={Q_B_INIT:.0e}  R={R_INIT:.0e}')

In [ ]:
# ── Load dataset ───────────────────────────────────────────────────────────────
_npz      = np.load(DATA_PATH)
Xv        = _npz['X_val'].astype(np.float32)   # (N_val_windows, WIN_LEN, 14) — raw IMU+GPS
Yv        = _npz['Y_val'].astype(np.float32)   # (N_val_windows, 3) — GPS vel, Y_iqr-normalised
vi        = _npz['val_valid_idx']               # (59,) sequence start indices
dv_iqr    = _npz['Y_iqr'].astype(np.float32)   # [1.516, 7.985, 0.1]
dv_median = _npz['Y_median'].astype(np.float32) # [-0.146, 0.820, 0.003]

N_VAL = len(vi)
print(f'Val sequences: {N_VAL}')
print(f'X_val shape: {Xv.shape}  (windows, WIN_LEN=200, channels=14)')
print(f'Y_iqr: {dv_iqr}  Y_median: {dv_median}')
print()

# IMU channel layout (channels 0–9 in X_val, same as MARSNet input)
# 0:3 = accel xyz (m/s²), 3:6 = gyro xyz (rad/s), 6:10 = other IMU
# channels 10:13 = GPS velocity (DO NOT pass to EKF during outage)
# channel 13 = outage flag

# Verify IMU channel values are in physical units (not normalised)
mid = WIN_LEN // 2
start0 = int(vi[0])
accel_sample = Xv[start0, mid, 0:3]
gyro_sample  = Xv[start0, mid, 3:6]
print(f'Sanity check — sequence 0, window {start0}, centre sample:')
print(f'  accel_xyz: {accel_sample}  (expect ~0-10 m/s² magnitude)')
print(f'  gyro_xyz:  {gyro_sample}  (expect ~0-1 rad/s magnitude)')
print(f'  |accel|={np.linalg.norm(accel_sample):.3f} m/s²  |gyro|={np.linalg.norm(gyro_sample):.4f} rad/s')

# Denormalise Y_val → physical GPS velocity (m/s)
# v_GPS_physical = Y_val * Y_iqr + Y_median
# CRITICAL: The EKF measurement model uses physical GPS velocity, not normalised.
print(f'\nY_val[0] (normalised): {Yv[start0]}')
print(f'Y_val[0] (physical):   {Yv[start0] * dv_iqr + dv_median} m/s')

In [ ]:
# ── Core KF implementation ─────────────────────────────────────────────────────

def build_F(dt=DT):
    """
    6x6 state transition matrix.
    State: [vx, vy, vz, bx, by, bz]
    v[t+1] = v[t] + (a_meas - b) * dt  →  F[v, b] = -dt * I
    b[t+1] = b[t]                       →  F[b, b] = I
    """
    F = np.eye(6, dtype=np.float64)
    F[0:3, 3:6] = -dt * np.eye(3)
    return F

def build_Q(Q_v, Q_bias):
    """6x6 process noise covariance."""
    return np.block([[Q_v    * np.eye(3), np.zeros((3, 3))],
                     [np.zeros((3, 3)),  Q_bias * np.eye(3)]]).astype(np.float64)

def build_H():
    """3x6 measurement matrix: measures velocity only."""
    return np.hstack([np.eye(3), np.zeros((3, 3))]).astype(np.float64)

def run_kf_sequence(seq_idx, Q_v, Q_bias, R_var,
                    Xv=Xv, Yv=Yv, vi=vi,
                    dv_iqr=dv_iqr, dv_median=dv_median,
                    SL=SEQ_LEN, dt=DT, return_paths=False):
    """
    Run Kalman Filter on one validation sequence.
    Returns endpoint drift in metres, and optionally the full path arrays.

    IMU: accelerometer from centre sample of each 200-sample window.
    GPS: available at every window EXCEPT during outage [os_, oe_).
    Position: integrated from velocity predictions, starting at outage onset.
    """
    start = int(vi[seq_idx])
    xrs   = Xv[start:start + SL]   # (SL, WIN_LEN, 14) — raw IMU windows
    yrs   = Yv[start:start + SL]   # (SL, 3) — GPS vel, Y_iqr-normalised
    if len(xrs) < SL:
        return (float('nan'), None, None) if return_paths else float('nan')

    # Denormalise GPS velocity to physical units
    v_gps = (yrs * dv_iqr + dv_median).astype(np.float64)  # (SL, 3) m/s

    # Accelerometer: centre sample of each window, channels 0:3
    accel = xrs[:, WIN_LEN // 2, 0:3].astype(np.float64)   # (SL, 3) m/s²

    # Outage window (same as MARSNet simulation)
    os_ = SL // 3
    oe_ = min(os_ + OE_LEN, SL)

    # ── Build KF matrices ────────────────────────────────────────────────────
    F = build_F(dt)
    Q = build_Q(Q_v, Q_bias)
    H = build_H()
    R = R_var * np.eye(3, dtype=np.float64)

    # ── Initialise state from first GPS reading and pre-outage accel bias ─────
    # The raw accelerometer includes gravity (~-9.8 m/s² on x-axis) and is not
    # gravity-compensated. Without attitude data we cannot rotate it out properly,
    # but since velocity is approximately constant before the outage (small net
    # acceleration), the pre-outage mean accelerometer reading is a good estimate
    # of the combined gravity + sensor bias the filter needs to subtract.
    BIAS_WINDOW = 20  # windows (2 seconds) immediately before outage
    pre_outage_bias = accel[max(0, os_ - BIAS_WINDOW):os_].mean(axis=0)

    x = np.zeros(6, dtype=np.float64)
    x[0:3] = v_gps[0]
    x[3:6] = pre_outage_bias
    P = np.eye(6, dtype=np.float64) * 1e-2
    P[3:6, 3:6] = np.eye(3) * 1e-1   # higher initial uncertainty on bias, since it's now an estimate not zero

    pos_pred = np.zeros((SL, 2), dtype=np.float64)
    pos_true = np.zeros((SL, 2), dtype=np.float64)
    vel_pred = np.zeros((SL, 3), dtype=np.float64)
    vel_pred[0] = x[0:3]

    for t in range(1, SL):
        a_meas = accel[t]
        u      = a_meas - x[3:6]
        x_pred = np.zeros(6)
        x_pred[0:3] = x[0:3] + u * dt
        x_pred[3:6] = x[3:6]
        P_pred = F @ P @ F.T + Q

        in_outage = (t >= os_) and (t < oe_)
        if not in_outage:
            z   = v_gps[t]
            inn = z - H @ x_pred
            S   = H @ P_pred @ H.T + R
            K   = P_pred @ H.T @ np.linalg.solve(S.T, np.eye(3)).T
            x   = x_pred + K @ inn
            P   = (np.eye(6) - K @ H) @ P_pred
        else:
            x = x_pred
            P = P_pred

        vel_pred[t] = x[0:3]

        # ── FIXED: only accumulate position during the outage window ──
        if t >= os_ and t < oe_:
            if t == os_:
                # Reset to zero at outage start — measure displacement from here
                pos_pred[t] = np.zeros(2)
                pos_true[t] = np.zeros(2)
            else:
                pos_pred[t] = pos_pred[t-1] + x[0:2] * dt
                pos_true[t] = pos_true[t-1] + v_gps[t, 0:2] * dt

    # Endpoint drift: displacement error at outage end
    drift = float(np.linalg.norm(pos_pred[oe_ - 1] - pos_true[oe_ - 1]))

    if return_paths:
        return drift, pos_pred, pos_true, vel_pred, v_gps, os_, oe_
    return drift

# ── Naive baseline (constant velocity from last GPS) ──────────────────────────
def run_naive_sequence(seq_idx, Xv=Xv, Yv=Yv, vi=vi,
                        dv_iqr=dv_iqr, dv_median=dv_median, SL=SEQ_LEN, dt=DT):
    start  = int(vi[seq_idx])
    yrs    = Yv[start:start + SL]
    if len(yrs) < SL: return float('nan')
    v_gps  = (yrs * dv_iqr + dv_median).astype(np.float64)
    os_    = SL // 3; oe_ = min(os_ + OE_LEN, SL)
    v_last = v_gps[os_ - 1] if os_ > 0 else np.zeros(3)
    pos_p  = np.zeros((SL, 2)); pos_t = np.zeros((SL, 2))
    for t in range(1, SL):
        if t >= os_ and t < oe_:
            if t == os_:
                pos_p[t] = np.zeros(2)
                pos_t[t] = np.zeros(2)
            else:
                pos_p[t] = pos_p[t-1] + v_last[0:2] * dt
                pos_t[t] = pos_t[t-1] + v_gps[t, 0:2] * dt
    return float(np.linalg.norm(pos_p[oe_ - 1] - pos_t[oe_ - 1]))

print('KF and naive functions defined.')
print(f'F matrix shape: {build_F().shape}  Q shape: {build_Q(1e-4,1e-6).shape}  H shape: {build_H().shape}')

In [ ]:
# ── Quick sanity check on sequence 0 before running all 59 ────────────────────
drift_s0, pp, pt, vp, vg, os_, oe_ = run_kf_sequence(
    0, Q_V_INIT, Q_B_INIT, R_INIT, return_paths=True)
naive_s0 = run_naive_sequence(0)

print(f'Sequence 0 sanity check (untuned KF):')
print(f'  KF drift:    {drift_s0:.3f}m')
print(f'  Naive drift: {naive_s0:.3f}m')
print(f'  Outage: [{os_}, {oe_})  ({(oe_-os_)*DT:.1f}s)')
print(f'  Pred vel at outage start: {vp[os_]} m/s')
print(f'  True vel at outage start: {vg[os_]} m/s')
print(f'  Pred pos at outage end:   {pp[oe_-1]}')
print(f'  True pos at outage end:   {pt[oe_-1]}')

# Quick plot
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(pt[:, 1], pt[:, 0], 'b-', lw=1.2, label='GPS truth')
axes[0].plot(pp[os_:oe_, 1], pp[os_:oe_, 0], 'r-', lw=1.2, label=f'KF ({drift_s0:.2f}m)')
axes[0].plot(pt[os_, 1], pt[os_, 0], 'ko', ms=5, label='Outage start')
axes[0].set_aspect('equal'); axes[0].legend(); axes[0].set_title('S0 — path')
axes[0].set_xlabel('East (m)'); axes[0].set_ylabel('North (m)')
axes[1].plot(vg[:, 0], 'b-', lw=0.8, label='GPS vx'); axes[1].plot(vp[:, 0], 'r--', lw=0.8, label='KF vx')
axes[1].plot(vg[:, 1], 'b-', lw=0.8, alpha=0.5, label='GPS vy'); axes[1].plot(vp[:, 1], 'r--', lw=0.8, alpha=0.5, label='KF vy')
axes[1].axvspan(os_, oe_, alpha=0.1, color='red', label='Outage')
axes[1].legend(fontsize=7); axes[1].set_title('S0 — velocity (m/s)')
axes[1].set_xlabel('Window index')
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'sanity_seq0.png'), dpi=120, bbox_inches='tight')
plt.show(); plt.close()
print('Sanity check passed — KF is producing sensible output.')

In [ ]:
# Diagnostic: check if pre-outage accel average looks like a constant gravity bias
for si in [0, 5, 41, 47]:
    start = int(vi[si])
    xrs = Xv[start:start+SEQ_LEN]
    accel = xrs[:, WIN_LEN//2, 0:3].astype(np.float64)
    os_ = SEQ_LEN // 3
    pre  = accel[:os_].mean(axis=0)
    post = accel[os_:os_+100].mean(axis=0)
    print(f'S{si}: pre-outage mean accel={pre}  post-outage mean accel={post}')

In [ ]:
# ── Run untuned KF on all 59 val sequences ─────────────────────────────────────
print('Running untuned KF (sensor-spec defaults)...')
t0 = time.time()

untuned_drifts = []
naive_drifts   = []
for si in range(N_VAL):
    d = run_kf_sequence(si, Q_V_INIT, Q_B_INIT, R_INIT)
    n = run_naive_sequence(si)
    untuned_drifts.append(d)
    naive_drifts.append(n)

print(f'Done in {time.time()-t0:.1f}s')

ud = np.array(untuned_drifts); nd = np.array(naive_drifts)
print(f'\nUntuned KF (59 sequences):')
print(f'  Mean={ud.mean():.2f}m  Median={np.median(ud):.2f}m  90th={np.percentile(ud,90):.2f}m')
print(f'  % under 5m: {100*np.mean(ud<5):.1f}%  % beats naive: {100*np.mean(ud<nd):.1f}%')
print(f'\n  ── Group breakdown (untuned) ──')
for grp in GROUP_ORDER:
    idx = [i for i in range(N_VAL) if GROUP_MAP.get(i) == grp]
    gm  = ud[idx].mean(); gn = nd[idx].mean()
    print(f'  {grp:<18}: {gm:.2f}m  (naive={gn:.2f}m)')
print(f'\n  Reference: MARSNet R20 = 6.56m  Naive = 5.96m')

In [ ]:
# ── Tune Q_v, Q_bias, R via Nelder-Mead on val set ────────────────────────────
# Optimise in log-space for numerical stability.
# This is the same protocol as the LAM_CVPRIOR grid search in MARSNet.

def objective(log_params):
    """Mean drift over all 59 val sequences. Nelder-Mead minimises this."""
    Q_v, Q_bias, R = np.exp(log_params)
    # Clamp to prevent degenerate values
    Q_v   = np.clip(Q_v,   1e-10, 10.)
    Q_bias = np.clip(Q_bias, 1e-12, 1.)
    R     = np.clip(R,     1e-10, 10.)
    drifts = [run_kf_sequence(si, Q_v, Q_bias, R) for si in range(N_VAL)]
    mean_d = np.nanmean(drifts)
    return mean_d

x0 = np.log([Q_V_INIT, Q_B_INIT, R_INIT])
print(f'Starting Nelder-Mead from: Q_v={Q_V_INIT:.0e}  Q_bias={Q_B_INIT:.0e}  R={R_INIT:.0e}')
print(f'x0 (log-space): {x0}')
print('Optimising...')
t0 = time.time()

result = minimize(
    objective,
    x0,
    method='Nelder-Mead',
    options={
        'maxiter': 800,
        'xatol':   1e-3,
        'fatol':   1e-3,
        'adaptive': True,    # adaptive simplex size — better for 3D
    },
    callback=lambda xk: print(f'  Q_v={np.exp(xk[0]):.2e}  Q_b={np.exp(xk[1]):.2e}  R={np.exp(xk[2]):.2e}  '
                               f'drift={objective(xk):.3f}m')
              if np.random.rand() < 0.05 else None  # print ~5% of iters
)

Q_V_OPT, Q_B_OPT, R_OPT = np.exp(result.x)
print(f'\nTuning complete in {time.time()-t0:.0f}s  ({result.nit} iters, converged={result.success})')
print(f'  Untuned: Q_v={Q_V_INIT:.2e}  Q_bias={Q_B_INIT:.2e}  R={R_INIT:.2e}')
print(f'  Tuned:   Q_v={Q_V_OPT:.2e}  Q_bias={Q_B_OPT:.2e}  R={R_OPT:.2e}')
print(f'  Final val drift (tuned): {result.fun:.3f}m')

In [ ]:
# ── Run tuned KF on all 59 val sequences ──────────────────────────────────────
print('Running tuned KF on all 59 val sequences...')
tuned_drifts  = []
tuned_paths   = {}

for si in range(N_VAL):
    drift, pp, pt, vp, vg, os_, oe_ = run_kf_sequence(
        si, Q_V_OPT, Q_B_OPT, R_OPT, return_paths=True)
    tuned_drifts.append(drift)
    # Save path data for paper figures (sequences 41, 47, 53)
    if si in [41, 47, 53]:
        tuned_paths[si] = {'pos_pred': pp, 'pos_true': pt,
                           'vel_pred': vp, 'vel_gps': vg,
                           'outage_start': os_, 'outage_end': oe_, 'drift_m': drift}

td = np.array(tuned_drifts); nd = np.array(naive_drifts)

print(f'\n{"="*60}')
print(f'  TUNED KF — Full 59-sequence evaluation')
print(f'{"="*60}')
print(f'  Mean:   {td.mean():.2f}m   (naive={nd.mean():.2f}m   R20=6.56m)')
print(f'  Median: {np.median(td):.2f}m  (naive={np.median(nd):.2f}m   R20=0.48m)')
print(f'  90th:   {np.percentile(td,90):.2f}m')
print(f'  % under 5m:   {100*np.mean(td<5):.1f}%  (R20=79.7%)')
print(f'  % beats naive:{100*np.mean(td<nd):.1f}%')
print()
print(f'  {"Group":<20} {"Tuned KF":>9} {"Untuned KF":>11} {"Naive":>7} {"MARSNet R20":>13}')
print(f'  {"-"*64}')
R20_REF = {'straight-short':0.38,'straight-med':2.28,'TURN':2.97,'FALSE-ALARM':32.56,'long-outage':24.52}
for grp in GROUP_ORDER:
    idx = [i for i in range(N_VAL) if GROUP_MAP.get(i) == grp]
    gt  = td[idx].mean(); gu = ud[idx].mean(); gn = nd[idx].mean()
    gr  = R20_REF.get(grp, float('nan'))
    print(f'  {grp:<20} {gt:>8.2f}m {gu:>10.2f}m {gn:>6.2f}m {gr:>12.2f}m')
print(f'{"="*60}')

# Per-sequence printout
print(f'\n  {"S":>3}  {"Tuned KF":>9}  {"Naive":>7}  Group')
print('  ' + '-'*40)
for si in range(N_VAL):
    grp = GROUP_MAP.get(si, '?')
    beat = '✓' if tuned_drifts[si] < naive_drifts[si] else '✗'
    print(f'  {si:>3}  {tuned_drifts[si]:>8.2f}m  {naive_drifts[si]:>6.2f}m  {grp} {beat}')

In [ ]:
# ── Save results CSVs (paper figures notebook reads these) ────────────────────
all_res_ekf = []
for si in range(N_VAL):
    all_res_ekf.append({
        'seq_idx':       si,
        'drift_m':       tuned_drifts[si],
        'naive_drift_m': naive_drifts[si],
        'untuned_drift': untuned_drifts[si],
        'group':         GROUP_MAP.get(si, 'unknown'),
        'outage_start':  SEQ_LEN // 3,
    })

df_ekf = pd.DataFrame(all_res_ekf)
csv_path = os.path.join(PLOT_DIR, 'val_seq_results.csv')
df_ekf.to_csv(csv_path, index=False)
print(f'Saved: {csv_path}')

# Save path arrays for paper figure 4 (sequences 41, 47, 53)
for si, pd_ in tuned_paths.items():
    path_file = os.path.join(PLOT_DIR, f'path_seq{si}.npy')
    np.save(path_file, {
        'pos_pred':      pd_['pos_pred'][:, :2],
        'pos_true':      pd_['pos_true'][:, :2],
        'outage_start':  pd_['outage_start'],
        'drift_m':       pd_['drift_m'],
        'naive_drift_m': naive_drifts[si],
    })
    print(f'  Saved path_seq{si}.npy  (drift={pd_["drift_m"]:.2f}m)')

# Save tuned parameters for reproducibility / paper appendix
params_file = os.path.join(PLOT_DIR, 'ekf_tuned_params.txt')
with open(params_file, 'w') as f:
    f.write(f'Q_v    = {Q_V_OPT:.6e}\n')
    f.write(f'Q_bias = {Q_B_OPT:.6e}\n')
    f.write(f'R      = {R_OPT:.6e}\n')
    f.write(f'val_mean_drift = {td.mean():.4f} m\n')
    f.write(f'nelder_mead_iters = {result.nit}\n')
    f.write(f'converged = {result.success}\n')
print(f'Tuned params saved: {params_file}')

In [ ]:
# ── Path grid: all 59 sequences (same layout as MARSNet grid) ─────────────────
def plot_path_grid(drifts, naive_drifts, title, save_path, n_cols=6):
    n  = N_VAL; nr = math.ceil(n / n_cols)
    fig, axes = plt.subplots(nr, n_cols, figsize=(n_cols*3.2, nr*2.8), squeeze=False)
    for si in range(n):
        ax  = axes[si // n_cols][si % n_cols]
        d,pp,pt,_,_,os_,oe_ = run_kf_sequence(si, Q_V_OPT, Q_B_OPT, R_OPT, return_paths=True)
        ax.plot(pt[:os_, 1], pt[:os_, 0], 'b-',  lw=1.)
        ax.plot(pt[os_:, 1], pt[os_:, 0], 'b--', lw=0.8, alpha=0.4)
        ax.plot(pp[os_:, 1], pp[os_:, 0], 'r-',  lw=1.)
        ax.plot(pt[os_, 1],  pt[os_, 0],  'ko',   ms=3, zorder=5)
        nd_  = naive_drifts[si]
        beat = d < nd_
        c    = 'green' if d < 5 else ('darkorange' if d < 15 else 'red')
        ax.set_title(f"S{si} {'✓' if beat else '✗'} {d:.1f}m", fontsize=7.5, color=c, fontweight='bold')
        ax.set_aspect('equal'); ax.tick_params(labelsize=5); ax.grid(True, alpha=0.25)
    for j in range(n, nr*n_cols): axes[j//n_cols][j%n_cols].set_visible(False)
    fig.suptitle(title, fontsize=10)
    plt.tight_layout(); plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show(); plt.close()

plot_path_grid(tuned_drifts, naive_drifts,
               'EKF Baseline (tuned) | Green<5m  Orange<15m  Red≥15m',
               os.path.join(PLOT_DIR, 'path_grid_all_seqs.png'))
print('Grid saved.')

In [ ]:
# ── Velocity comparison plots for 3 representative sequences ──────────────────
# S41 (turn), S47 (false-alarm), S53 (long outage)
# These go in the supplementary / qualitative section.

TARGET_SEQS = [(41, 'S41 — Turn'), (47, 'S47 — False-alarm'), (53, 'S53 — Long outage')]
fig, axes = plt.subplots(len(TARGET_SEQS), 2, figsize=(11, 8))

for row, (si, label) in enumerate(TARGET_SEQS):
    d, pp, pt, vp, vg, os_, oe_ = run_kf_sequence(
        si, Q_V_OPT, Q_B_OPT, R_OPT, return_paths=True)
    nd_ = naive_drifts[si]

    # Left: path
    ax = axes[row][0]
    ax.plot(pt[:os_, 1], pt[:os_, 0],  'k-',   lw=1.5, label='GPS truth (aided)')
    ax.plot(pt[os_:, 1], pt[os_:, 0], 'k--',   lw=1.,  alpha=0.4, label='GPS truth (outage)')
    ax.plot(pp[os_:, 1], pp[os_:, 0], 'r-',    lw=1.5, label=f'EKF ({d:.1f}m)')
    ax.plot(pt[os_, 1],  pt[os_, 0],  'ko',    ms=5, zorder=5, label='Outage start')
    ax.set_aspect('equal'); ax.legend(fontsize=7); ax.grid(True, alpha=0.3)
    ax.set_title(f'{label} — Path (drift={d:.2f}m  naive={nd_:.2f}m)', fontsize=8)
    ax.set_xlabel('East (m)'); ax.set_ylabel('North (m)')

    # Right: velocity traces
    ax2 = axes[row][1]
    t_ax = np.arange(SEQ_LEN) * DT
    ax2.plot(t_ax, vg[:, 0], 'b-',  lw=1., label='GPS vx')
    ax2.plot(t_ax, vp[:, 0], 'b--', lw=1., label='KF vx')
    ax2.plot(t_ax, vg[:, 1], 'r-',  lw=1., label='GPS vy')
    ax2.plot(t_ax, vp[:, 1], 'r--', lw=1., label='KF vy')
    ax2.axvspan(os_*DT, oe_*DT, alpha=0.12, color='grey', label='Outage')
    ax2.legend(fontsize=7, ncol=3); ax2.grid(True, alpha=0.3)
    ax2.set_xlabel('Time (s)'); ax2.set_ylabel('Velocity (m/s)')
    ax2.set_title(f'{label} — Velocity', fontsize=8)

plt.suptitle('EKF Baseline — Representative sequences', fontsize=9, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'velocity_examples.png'), dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print('Velocity plots saved.')

In [ ]:
# ── Covariance growth analysis (S53–S58 long outage) ──────────────────────────
# Shows how P grows unboundedly without GPS updates — a key paper finding.
# Contrasts with MARSNet which doesn't track uncertainty explicitly.

print('Covariance growth during outage (S53 — 30s outage):')
si   = 53
start = int(vi[si])
xrs   = Xv[start:start+SEQ_LEN]
yrs   = Yv[start:start+SEQ_LEN]
v_gps_phys = (yrs * dv_iqr + dv_median).astype(np.float64)
accel = xrs[:, WIN_LEN//2, 0:3].astype(np.float64)
os_  = SEQ_LEN // 3; oe_ = min(os_ + OE_LEN, SEQ_LEN)

F = build_F(); Q = build_Q(Q_V_OPT, Q_B_OPT); H = build_H()
R = R_OPT * np.eye(3)
x = np.zeros(6); x[0:3] = v_gps_phys[0]; P = np.eye(6) * 1e-2
P_trace = []
for t in range(1, SEQ_LEN):
    u = accel[t] - x[3:6]
    x_pred = np.zeros(6); x_pred[0:3] = x[0:3] + u*DT; x_pred[3:6] = x[3:6]
    P_pred = F @ P @ F.T + Q
    if t < os_ or t >= oe_:
        z = v_gps_phys[t]; inn = z - H @ x_pred
        S = H @ P_pred @ H.T + R
        K = P_pred @ H.T @ np.linalg.solve(S.T, np.eye(3)).T
        x = x_pred + K @ inn; P = (np.eye(6) - K @ H) @ P_pred
    else:
        x = x_pred; P = P_pred
    P_trace.append(np.trace(P[:3, :3]))   # velocity covariance trace

fig, ax = plt.subplots(figsize=(7, 2.8))
t_ax = np.arange(1, SEQ_LEN) * DT
ax.plot(t_ax, P_trace, 'b-', lw=1.2)
ax.axvspan(os_*DT, oe_*DT, alpha=0.15, color='red', label='GPS outage')
ax.set_xlabel('Time (s)'); ax.set_ylabel('tr(P_v) — velocity uncertainty')
ax.set_title('Covariance growth during GPS outage (S53, 30s outage)')
ax.legend(fontsize=8)
print(f'  P_v trace at outage start: {P_trace[os_-1]:.6f}')
print(f'  P_v trace at outage end:   {P_trace[oe_-2]:.6f}')
print(f'  Growth factor: {P_trace[oe_-2]/(P_trace[os_-1]+1e-12):.1f}x')
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'covariance_growth.png'), dpi=120, bbox_inches='tight')
plt.show(); plt.close()

In [ ]:
# ── Final summary (paper-ready numbers) ───────────────────────────────────────
print('=' * 65)
print('  EKF BASELINE — FINAL RESULTS')
print('=' * 65)
print(f'  Tuned parameters:')
print(f'    Q_v    = {Q_V_OPT:.3e}  (velocity process noise)')
print(f'    Q_bias = {Q_B_OPT:.3e}  (bias random walk noise)')
print(f'    R      = {R_OPT:.3e}   (GPS measurement noise)')
print()
print(f'  {"Metric":<28}  {"Untuned":>9}  {"Tuned":>9}  {"Naive":>9}  {"MARSNet R20":>12}')
print(f'  {"-"*73}')
print(f'  {"Mean drift (m)":<28}  {ud.mean():>9.2f}  {td.mean():>9.2f}  {nd.mean():>9.2f}  {6.56:>12.2f}')
print(f'  {"Median drift (m)":<28}  {np.median(ud):>9.2f}  {np.median(td):>9.2f}  {np.median(nd):>9.2f}  {0.48:>12.2f}')
print(f'  {"90th pctile (m)":<28}  {np.percentile(ud,90):>9.2f}  {np.percentile(td,90):>9.2f}  {"—":>9}  {32.52:>12.2f}')
print(f'  {"% under 5m":<28}  {100*np.mean(ud<5):>8.1f}%  {100*np.mean(td<5):>8.1f}%  {"—":>9}  {79.7:>11.1f}%')
print()
print(f'  {"Group":<20}  {"Untuned":>9}  {"Tuned":>9}  {"Naive":>9}  {"MARSNet R20":>12}')
print(f'  {"-"*65}')
R20_REF = {'straight-short':0.38,'straight-med':2.28,'TURN':2.97,'FALSE-ALARM':32.56,'long-outage':24.52}
for grp in GROUP_ORDER:
    idx = [i for i in range(N_VAL) if GROUP_MAP.get(i) == grp]
    gu = ud[idx].mean(); gt = td[idx].mean(); gn = nd[idx].mean(); gr = R20_REF.get(grp, float('nan'))
    print(f'  {grp:<20}  {gu:>9.2f}  {gt:>9.2f}  {gn:>9.2f}  {gr:>12.2f}')
print('=' * 65)
print()
print('  PAPER NARRATIVE CHECK (Expected pattern):')
idx_turn = [i for i in range(N_VAL) if GROUP_MAP.get(i) == 'TURN']
idx_fa   = [i for i in range(N_VAL) if GROUP_MAP.get(i) == 'FALSE-ALARM']
turn_win = td[idx_turn].mean() < 10.   # EKF should struggle on turns
fa_win   = td[idx_fa].mean() < 5.     # EKF should excel on false-alarm
print(f'  EKF on turns (S41-46): {td[idx_turn].mean():.2f}m  {"✓ degraded (expected)" if not turn_win else "~ OK"}')
print(f'  EKF on false-alarm (S47-52): {td[idx_fa].mean():.2f}m  {"✓ low (expected)" if fa_win else "~ not as expected"}')
print(f'  MARSNet on turns: 2.97m  EKF on turns: {td[idx_turn].mean():.2f}m')
print(f'  MARSNet on false-alarm: 32.56m  EKF on false-alarm: {td[idx_fa].mean():.2f}m')
print()
print('  If EKF struggles on turns but succeeds on false-alarm, the 3-way narrative holds.')
print('  → Paper: naive fails turns, EKF fails turns, MARSNet succeeds turns')
print('           EKF & naive succeed false-alarm, MARSNet fails false-alarm')
print('  This makes a clean complementary strengths story.')

In [ ]:
# Targeted diagnostic: does S0 give the same drift whether called alone or via the eval loop?
d_direct = run_kf_sequence(0, Q_V_INIT, Q_B_INIT, R_INIT)
print(f'S0 direct call: {d_direct:.3f}m')

# Now replicate exactly what the eval loop in Cell 8 does
for si in range(3):
    d = run_kf_sequence(si, Q_V_INIT, Q_B_INIT, R_INIT)
    n = run_naive_sequence(si)
    print(f'S{si}: KF={d:.3f}m  naive={n:.3f}m')

# Check: is GROUP_MAP correct? does straight-short really map to 0-34?
print()
print('GROUP_MAP[0]:', GROUP_MAP.get(0))
print('GROUP_MAP[41]:', GROUP_MAP.get(41))
print('GROUP_MAP[47]:', GROUP_MAP.get(47))

# Check S41 and S47 directly with the patched function
d41 = run_kf_sequence(41, Q_V_INIT, Q_B_INIT, R_INIT)
d47 = run_kf_sequence(47, Q_V_INIT, Q_B_INIT, R_INIT)
print(f'\nS41 (TURN) direct: {d41:.3f}m')
print(f'S47 (FALSE-ALARM) direct: {d47:.3f}m')

In [ ]:
# Check variance of accel during the pre-outage window — high variance means
# the "near-constant velocity" assumption behind the bias trick is violated
for si in [0, 1, 2, 41, 47]:
    start = int(vi[si])
    xrs = Xv[start:start+SEQ_LEN]
    accel = xrs[:, WIN_LEN//2, 0:3].astype(np.float64)
    os_ = SEQ_LEN // 3
    pre = accel[:os_]
    print(f'S{si}: pre-outage accel std={pre.std(axis=0)}  '
          f'mean={pre.mean(axis=0)}  '
          f'first-half mean={pre[:os_//2].mean(axis=0)}  '
          f'second-half mean={pre[os_//2:].mean(axis=0)}')

# Also check: what does the bias estimate look like vs the TRUE bias needed
# (estimate true bias as: accel during outage minus the acceleration implied by GPS velocity change)
for si in [41, 47]:
    start = int(vi[si])
    xrs = Xv[start:start+SEQ_LEN]
    yrs = Yv[start:start+SEQ_LEN]
    accel = xrs[:, WIN_LEN//2, 0:3].astype(np.float64)
    v_gps = (yrs * dv_iqr + dv_median).astype(np.float64)
    os_ = SEQ_LEN // 3; oe_ = os_ + 100
    # true acceleration implied by GPS velocity change during outage
    true_accel = np.diff(v_gps[os_:oe_], axis=0) / DT
    measured_accel = accel[os_:oe_-1]
    implied_bias = measured_accel - true_accel
    print(f'\nS{si}: bias used (pre-outage mean) = {accel[:os_].mean(axis=0)}')
    print(f'S{si}: implied true bias during outage (mean) = {implied_bias.mean(axis=0)}')
    print(f'S{si}: implied true bias during outage (std)  = {implied_bias.std(axis=0)}')

In [ ]:
# Verify which bias window is actually being used right now
import inspect
print(inspect.getsource(run_kf_sequence).split('pre_outage_bias')[1][:200])

In [ ]:
# Compare bias estimate: full pre-outage window vs last 20 windows
for si in [0, 41, 47]:
    start = int(vi[si])
    xrs = Xv[start:start+SEQ_LEN]
    accel = xrs[:, WIN_LEN//2, 0:3].astype(np.float64)
    os_ = SEQ_LEN // 3
    full_bias  = accel[:os_].mean(axis=0)
    short_bias = accel[max(0, os_-20):os_].mean(axis=0)
    print(f'S{si}: full(100w)={full_bias}  short(20w)={short_bias}  diff={short_bias-full_bias}')